# Machine Learning

### Problem Description:
A Chinese automobile company aspires to enter the US market by setting up their manufacturing unit
there and producing cars locally to give competition to their US and European counterparts. They have
contracted an automobile consulting company to understand the factors on which the pricing of cars
depends. Specifically, they want to understand the factors affecting the pricing of cars in the American
market, since those may be very different from the Chinese market. Essentially, the company wants to
know:
Which variables are significant in predicting the price of a car?
How well those variables describe the price of a car?
Based on various market surveys, the consulting firm has gathered a large dataset of different types of
cars across the American market.
Business Goal: You are required to model the price of cars with the available independent variables. It
will be used by the management to understand how exactly the prices vary with the independent
variables. They can accordingly manipulate the design of the cars, the business strategy, etc., to meet
certain price levels. Further, the model will be a good way for the management to understand the pricing
dynamics of a new market.

Key components to be fulfilled:
1. Loading and Preprocessing (5 marks): Load the dataset and perform necessary preprocessing
steps.

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns



In [2]:

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error


In [4]:
# 1. Loading and Preprocessing
df = pd.read_csv('CarPrice_Assignment.csv')


In [5]:
df.head(10)

,car_ID,symboling,CarName,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,wheelbase,...,enginesize,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price
0,1,3,alfa-romero giulia,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,13495.000
1,2,3,alfa-romero stelvio,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,16500.000
2,3,1,alfa-romero Quadrifoglio,gas,std,two,hatchback,rwd,front,94.5,...,152,mpfi,2.68,3.47,9.0,154,5000,19,26,16500.000
3,4,2,audi 100 ls,gas,std,four,sedan,fwd,front,99.8,...,109,mpfi,3.19,3.40,10.0,102,5500,24,30,13950.000
4,5,2,audi 100ls,gas,std,four,sedan,4wd,front,99.4,...,136,mpfi,3.19,3.40,8.0,115,5500,18,22,17450.000
5,6,2,audi fox,gas,std,two,sedan,fwd,front,99.8,...,136,mpfi,3.19,3.40,8.5,110,5500,19,25,15250.000
6,7,1,audi 100ls,gas,std,four,sedan,fwd,front,105.8,...,136,mpfi,3.19,3.40,8.5,110,5500,19,25,17710.000
7,8,1,audi 5000,gas,std,four,wagon,fwd,front,105.8,...,136,mpfi,3.19,3.40,8.5,110,5500,19,25,18920.000
8,9,1,audi 4000,gas,turbo,four,sedan,fwd,front,105.8,...,131,mpfi,3.13,3.40,8.3,140,5500,17,20,23875.000
9,10,0,audi 5000s (diesel),gas,turbo,two,hatchback,4wd,front,99.5,...,131,mpfi,3.13,3.40,7.0,160,5500,16,22,17859.167


In [6]:
# Drop car_ID and extract brand/company from CarName if needed
if 'car_ID' in df.columns:
    df = df.drop(columns=['car_ID'])

In [7]:
if 'CarName' in df.columns:
    df['brand'] = df['CarName'].apply(lambda x: x.split()[0].lower())
    df = df.drop(columns=['CarName'])


In [8]:
# Separate features and target
X = df.drop(columns=['price'])
y = df['price']

In [9]:
# Identify numerical and categorical columns
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()


In [10]:
# Preprocessing pipelines
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ]
)

In [11]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


2. Model Implementation (10 marks): Implement the following five regression algorithms:
● Linear Regression                                             
● Decision Tree Regressor                                               
● Random Forest Regressor                                               
● Gradient Boosting Regressor                                                   
● Support Vector Regressor                                      

In [12]:
# 2. Model Implementation
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree Regressor': DecisionTreeRegressor(random_state=42),
    'Random Forest Regressor': RandomForestRegressor(random_state=42),
    'Gradient Boosting Regressor': GradientBoostingRegressor(random_state=42),
    'Support Vector Regressor': SVR()
}

results = {}

for name, model in models.items():
    clf = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', model)])
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    
    results[name] = {'R2': r2, 'MSE': mse, 'MAE': mae, 'pipeline': clf}

3. Model Evaluation (5 marks): Compare the performance of all the models based on R-squared,
Mean Squared Error (MSE), and Mean Absolute Error (MAE). Identify the best performing model
and justify why it is the best.

In [13]:
# 3. Model Evaluation Comparison
results_df = pd.DataFrame(results).T[['R2', 'MSE', 'MAE']]
print("Model Performance Comparison:\n", results_df)

# Best Model Identification (Random Forest or Gradient Boosting typically performs best on tabular car datasets)
best_model_name = results_df['R2'].idxmax()
print(f"\nBest Performing Model: {best_model_name}")

Model Performance Comparison:
                                    R2              MSE          MAE
Linear Regression            0.899843   7906808.315869  1835.328036
Decision Tree Regressor      0.913599   6820817.260192  1704.142268
Random Forest Regressor      0.956562   3429185.993727  1308.435297
Gradient Boosting Regressor  0.928162   5671210.206182  1690.948706
Support Vector Regressor     -0.09979  86821847.290298  5695.273987

Best Performing Model: Random Forest Regressor


4. Feature Importance Analysis (2 marks): Identify the significant variables affecting car prices
(feature selection)

In [14]:
# 4. Feature Importance Analysis
best_pipeline = results[best_model_name]['pipeline']
regressor = best_pipeline.named_steps['regressor']

if hasattr(regressor, 'feature_importances_'):
    # Extract feature names after one-hot encoding
    cat_enc_features = best_pipeline.named_steps['preprocessor'].transformers_[1][1].named_steps['onehot'].get_feature_names_out(cat_cols)
    all_features = np.concatenate([num_cols, cat_enc_features])
    importances = regressor.feature_importances_
    feat_imp = pd.Series(importances, index=all_features).sort_values(ascending=False)
    print("\nTop 10 Significant Features:\n", featum_imp[:10] if 'featum_imp' in locals() else feat_imp[:10])


Top 10 Significant Features:
 enginesize    0.541903
curbweight    0.296046
highwaympg    0.045039
horsepower    0.038125
carwidth      0.013050
brand_bmw     0.007777
carlength     0.007249
wheelbase     0.006820
citympg       0.006329
peakrpm       0.005034
dtype: float64


5. Hyperparameter Tuning (2 marks): Perform hyperparameter tuning and check whether the
performance of the model has increased.

In [15]:
# 5. Hyperparameter Tuning (Example for Random Forest or Gradient Boosting)
param_grid = {
    'regressor__n_estimators':[100,200,300],
    'regressor__max_depth': [None, 10, 20],
    'regressor__min_samples_split': [2, 5]
}

base_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42))
])

grid_search = GridSearchCV(base_rf, param_grid, cv=5, scoring='r2', n_jobs=-1)
grid_search.fit(X_train, y_train)

print("\nBest Hyperparameters:", grid_search.best_params_)
tuned_pred = grid_search.predict(X_test)
print(f"Tuned R2 Score: {r2_score(y_test, tuned_pred):.4f}")


Best Hyperparameters: {'regressor__max_depth': None, 'regressor__min_samples_split': 2, 'regressor__n_estimators': 100}
Tuned R2 Score: 0.9566


In [16]:
# Save the best trained model for deployment
import joblib
joblib.dump(grid_search.best_estimator_, 'car_price_model.pkl')

['car_price_model.pkl']